In [1]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import sqlite3
import hashlib
import json
import xml.etree.ElementTree as ET


class DBManager:
    def __init__(self):
        self.conn = sqlite3.connect("library_v2.db")
        self.cursor = self.conn.cursor()
        self.cursor.execute(
            'CREATE TABLE IF NOT EXISTS authors (id INTEGER PRIMARY KEY, name TEXT, country TEXT, years TEXT, birth_year INTEGER)')
        self.cursor.execute(
            'CREATE TABLE IF NOT EXISTS books (id INTEGER PRIMARY KEY, author_id INTEGER, title TEXT, pages INTEGER, publisher TEXT, year INTEGER)')
        self.cursor.execute('CREATE TABLE IF NOT EXISTS users (username TEXT PRIMARY KEY, password_hash TEXT)')
        h = hashlib.md5("admin".encode()).hexdigest()
        self.cursor.execute("INSERT OR IGNORE INTO users VALUES (?, ?)", ("admin", h))
        self.conn.commit()

    def authenticate(self, user, pwd):
        h = hashlib.md5(pwd.encode()).hexdigest()
        self.cursor.execute("SELECT * FROM users WHERE username=? AND password_hash=?", (user, h))
        return self.cursor.fetchone() is not None

    def get_authors(self):
        self.cursor.execute("SELECT id, name, country, years FROM authors")
        return self.cursor.fetchall()

    def get_books(self):
        self.cursor.execute("SELECT * FROM books")
        return self.cursor.fetchall()

    def add_author(self, name, country, years):
        by = int(years.split('-')[0]) if '-' in years else (int(years) if years.isdigit() else 0)
        self.cursor.execute("INSERT INTO authors (name, country, years, birth_year) VALUES (?,?,?,?)",
                            (name, country, years, by))
        self.conn.commit()

    def add_book(self, aid, title, pages, pub, year):
        self.cursor.execute("INSERT INTO books (author_id, title, pages, publisher, year) VALUES (?,?,?,?,?)",
                            (aid, title, pages, pub, year))
        self.conn.commit()


class LoginWindow(tk.Toplevel):
    def __init__(self, parent, db, on_success):
        super().__init__(parent)
        self.db, self.on_success = db, on_success
        self.title("Авторизация")
        self.geometry("300x200")
        tk.Label(self, text="Логин:").pack(pady=5)
        self.u = tk.Entry(self);
        self.u.pack()
        tk.Label(self, text="Пароль:").pack(pady=5)
        self.p = tk.Entry(self, show="*");
        self.p.pack()
        tk.Button(self, text="Войти", command=self.login, width=15).pack(pady=20)

    def login(self):
        if self.db.authenticate(self.u.get(), self.p.get()):
            self.on_success();
            self.destroy()
        else:
            messagebox.showerror("Ошибка", "Неверный логин или пароль")


class LibraryApp:
    def __init__(self, root, db):
        self.root, self.db = root, db
        self.root.title("Информационная система Библиотеки")
        self.root.geometry("800x500")
        tabs = ttk.Notebook(root)
        self.t1, self.t2 = ttk.Frame(tabs), ttk.Frame(tabs)
        tabs.add(self.t1, text="Авторы");
        tabs.add(self.t2, text="Книги")
        tabs.pack(expand=1, fill="both")
        self.setup_ui();
        self.refresh()

    def setup_ui(self):
        cols_a = ("ID", "Имя", "Страна", "Годы")
        self.tree_a = ttk.Treeview(self.t1, columns=cols_a, show='headings')
        for c in cols_a: self.tree_a.heading(c, text=c)
        self.tree_a.pack(expand=1, fill="both", padx=5, pady=5)

        btns_a = ttk.Frame(self.t1)
        btns_a.pack(fill="x", padx=5, pady=5)
        ttk.Button(btns_a, text="Добавить автора", command=self.add_a).pack(side="left", padx=2)
        ttk.Button(btns_a, text="Импорт JSON/XML", command=self.import_f).pack(side="left", padx=2)
        ttk.Button(btns_a, text="Экспорт JSON", command=self.export_f).pack(side="left", padx=2)

        cols_b = ("ID", "AID", "Название", "Стр", "Изд", "Год")
        self.tree_b = ttk.Treeview(self.t2, columns=cols_b, show='headings')
        for c in cols_b: self.tree_b.heading(c, text=c)
        self.tree_b.pack(expand=1, fill="both", padx=5, pady=5)
        ttk.Button(self.t2, text="Добавить книгу", command=self.add_b).pack(pady=5)

    def refresh(self):
        for t, data in [(self.tree_a, self.db.get_authors()), (self.tree_b, self.db.get_books())]:
            for i in t.get_children(): t.delete(i)
            for r in data: t.insert("", "end", values=r)

    def add_a(self):
        w = tk.Toplevel(self.root)
        w.title("Автор")
        tk.Label(w, text="Имя").pack();
        e1 = tk.Entry(w);
        e1.pack()
        tk.Label(w, text="Страна").pack();
        e2 = tk.Entry(w);
        e2.pack()
        tk.Label(w, text="Годы").pack();
        e3 = tk.Entry(w);
        e3.pack()

        def save(): self.db.add_author(e1.get(), e2.get(), e3.get()); self.refresh(); w.destroy()

        tk.Button(w, text="Сохранить", command=save).pack(pady=10)

    def add_b(self):
        w = tk.Toplevel(self.root)
        w.title("Книга")
        tk.Label(w, text="ID Автора").pack();
        e1 = tk.Entry(w);
        e1.pack()
        tk.Label(w, text="Название").pack();
        e2 = tk.Entry(w);
        e2.pack()
        tk.Label(w, text="Страниц").pack();
        e3 = tk.Entry(w);
        e3.pack()
        tk.Label(w, text="Изд").pack();
        e4 = tk.Entry(w);
        e4.pack()
        tk.Label(w, text="Год").pack();
        e5 = tk.Entry(w);
        e5.pack()

        def save(): self.db.add_book(e1.get(), e2.get(), e3.get(), e4.get(), e5.get()); self.refresh(); w.destroy()

        tk.Button(w, text="Сохранить", command=save).pack(pady=10)

    def export_f(self):
        s = self.tree_a.selection()
        if not s: return
        v = self.tree_a.item(s[0])['values']
        d = {"name": v[1], "country": v[2], "years": v[3]}
        p = filedialog.asksaveasfilename(defaultextension=".json")
        if p:
            with open(p, 'w', encoding='utf-8') as f: json.dump(d, f, ensure_ascii=False, indent=4)

    def import_f(self):
        p = filedialog.askopenfilename()
        if not p: return
        try:
            if p.endswith('.json'):
                with open(p, 'r', encoding='utf-8') as f:
                    d = json.load(f);
                    n, c, y = d['name'], d['country'], str(d['years'])
            else:
                r = ET.parse(p).getroot();
                n, c = r.find('name').text, r.find('country').text
                yn = r.find('years');
                y = f"{yn.get('born')}-{yn.get('died')}"
            self.db.add_author(n, c, y);
            self.refresh()
        except:
            messagebox.showerror("Ошибка", "Некорректный файл")


if __name__ == "__main__":
    db_m = DBManager();
    root = tk.Tk();
    root.withdraw()
    LoginWindow(root, db_m, lambda: (root.deiconify(), LibraryApp(root, db_m)))
    root.mainloop()

ModuleNotFoundError: No module named 'PyQt6'